# coAgent — free-T4 smoke test (acceptance test AT-1)

This notebook is the **release gate for v0.1**. Everything provable on a CPU is
already proven by the 476-test suite; what no local test can reach is the real
GPU path: an NF4 4-bit load of a 7B model on a 15 GB T4, a live generation loop,
and a verify-gated commit driven by weights instead of a scripted backend.

Work top to bottom. Each step states what a pass looks like and what to do when
it fails. Do not skip step 2 — it is the only cheap failure here.

**Runtime → Change runtime type → T4 GPU** before you start.

## 1 · Install

A cold install pulls torch and transformers; this is the slowest cell in the
notebook (~3–5 min). `co-agent-coder` is the distribution, `coagent` is the import.

In [ ]:
!pip -q install "co-agent-coder @ git+https://github.com/lyle-mlengineer/coagent"

import coagent
print("coagent", coagent.__version__)

## 2 · Refuse early if the runtime is wrong

Checks T4 / compute capability 7.5 / ≥14 GB VRAM **without loading weights**.
A failure here is the good outcome: it is cheap.

Expected: `Tesla T4 · 15 GB · CC 7.5` and verdict `OK`.

In [ ]:

from coagent.llm import preflight_gpu

check = preflight_gpu()
print(check.describe())
print("verdict:", "OK - a free-tier T4" if check.ok else "REFUSED")
if not check.ok:
    print("\n" + check.reason)
    raise SystemExit("fix the runtime before continuing")


## 3 · A toy repo with one failing test

Small on purpose: the point is the loop, not the repo. `add()` subtracts, and
the test asserts it adds. AT-1 is this exact repair.

In [ ]:

import os, subprocess

os.makedirs("/content/demo", exist_ok=True)
os.chdir("/content/demo")

CALC = "def add(a, b):\n    return a - b\n"
TEST = (
    "from calc import add\n"
    "\n"
    "def test_add():\n"
    "    assert add(2, 3) == 5\n"
)
PYPROJECT = '[project]\nname = "demo"\nversion = "0.1"\n'

open("calc.py", "w").write(CALC)
os.makedirs("tests", exist_ok=True)
open("tests/test_calc.py", "w").write(TEST)
open("pyproject.toml", "w").write(PYPROJECT)

if not os.path.isdir(".git"):
    subprocess.run("git init -q", shell=True, check=True)
    subprocess.run("git add -A", shell=True, check=True)
    subprocess.run(
        'git -c user.email=a@b.c -c user.name=t commit -qm "init"',
        shell=True, check=True,
    )

out = subprocess.run("python -m pytest -q", shell=True, capture_output=True, text=True)
print(out.stdout[-300:])


## 4 · Load the model (the expensive step)

NF4 4-bit, fp16 compute, SDPA attention — Turing has no bf16 and no
FlashAttention-2. Cold load is ~2–4 min; weights cache for later runs.

Expected: no `OutOfMemoryError`, and `nvidia-smi` shows ~4–6 GB used.

In [ ]:
import time, torch

from coagent import coAgent

t0 = time.time()
agent = coAgent(repo=".", auto=True)   # auto=True: non-interactive approvals deny
print(f"loaded in {time.time() - t0:.1f}s · {agent.model_id}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 5 · AT-1: fix the failing test, end to end

The run drives SURVEY → PLAN → ACT → VERIFY → CHECKPOINT → REPORT on real
weights. It is done when `run_tests` passes — not when the model says so.

Pass looks like: a commit on a `coagent/<session>` branch with a
`Coagent-Verify: passed` trailer, and the test green afterwards.

**A 7B model may not solve this on the first try.** That is information, not a
crash: capture `agent.show_log()`. What matters for the gate is that the loop
reports honestly and never commits a failing run as verified.

In [ ]:
result = agent.chat("fix add() so the tests pass")
print(result.render() if hasattr(result, "render") else result)
print(agent.status_line())

## 6 · Prove the branch, not the claim

Never trust the summary. Read the trailers and re-run the test yourself.

In [ ]:
print(agent.show_log(last=8))
print("branch:", agent.branch())
print(subprocess.run("git log --oneline -5", shell=True, capture_output=True, text=True).stdout)
print(subprocess.run("git log -1 --format='%(trailers)'", shell=True, capture_output=True, text=True).stdout)
print(subprocess.run("python -m pytest -q", shell=True, capture_output=True, text=True).stdout[-200:])

## 7 · Durability: simulate the VM dying

Colab reclaims runtimes without warning. `resume()` must rebuild the briefing
from git alone — no model call, no in-memory state — because the repository is
the memory, not the VM.

In [ ]:

session = agent.session_id
agent.close()
print("closed; simulating a fresh VM...")
print()

agent2 = coAgent(repo=".", auto=True, session_id=session)
print(agent2.resume())


## 8 · Verdict

Record the outcome here — this is the release evidence.

| Check | Pass criteria |
| --- | --- |
| 2 · Preflight | T4 / CC 7.5 / ≥14 GB |
| 4 · Model load | no OOM, ~4–6 GB VRAM |
| 5 · Loop runs | phases advance, no crash |
| 6 · Honest commit | `Coagent-Verify` matches reality |
| 7 · Resume | briefing rebuilt from git alone |

If the loop stalls, check `agent.show_log()` before assuming a model problem:
an approval denied by `auto=True` looks like a stall and is not.